<a href="https://colab.research.google.com/github/Admindatosgobes/Laboratorio-de-Datos/blob/main/Magnitudes_Geograficas_en_KeplerGL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crea tu mapa de magnitudes geográficas en KeplerGL

---



## 0. Introducción

En este ejercicio visualizaremos información georreferenciada relacionada con la actividad sísmica de la erupción del volcán de la Palma en torno a septiembre de 2021.

En este ejercicio vamos a utilizar datos abiertos del Cabildo Insular de La Palma recopilados durante la actividad sísmica anterior y posterior a la erupción volcánica en La Palma en 2021, y que están disponibles aquí:  

https://datos.gob.es/es/catalogo/l03380010-terremotos

En este dataset encontramos el registro de cada uno de los puntos en los que se detectó actividad sísmica durante esos días.

Para la creación del mapa nos centraremos en la variable asociada a la actividad sísmica: magnitud, así como la longitud y latitud de cada punto y la fecha y la hora de cada evento.

## 1. Creación de un Dataframe

Para la lectura de los datos utilizaremos la libreria **Pandas** de Python, que nos permite estructurar los datos en filas y columnas de forma intuitiva y operar sobre ellos.  

Igualmente, y dado que tenemos la fecha en formato de año, mes y día de la semana así como la hora de los terremotos necesitaremos una librería para modificar y crear formatos de fecha que podamos manejar fácilmente. Para ello disponemos de la librería **Datetime** en Python.

La libreria **IO** y **files** de Google Colab nos permite hacer una lectura de datos desde otras localizaciones fuera de este notebook.

In [1]:
import pandas as pd
import numpy as np
import datetime
import io
from google.colab import files
import warnings
warnings.filterwarnings("ignore")

El primer paso será subir al notebook el fichero CSV con los datos de los terremotos en la isla de La Palma en 2021 que encontramos en el repositorio de Github.

In [2]:
uploaded = files.upload()

Saving terremotos.csv to terremotos.csv


Procedemos entonces a la lectura del fichero de entrada en forma de pandas a partir de un fichero en formato .csv.

Una vez importados los datos y almacenados en un dataframe podemos ver el nombre de las variables y su tipo a través del comando `.info()` de Pandas, que nos ofrece:

*   **Número de entradas** en el dataframe
*   **Nombre de las columnas** asociada con cada variable
*   **Tipo de cada variable**, incluyendo object o float.

In [5]:
content = pd.read_csv(io.BytesIO(uploaded['terremotos.csv']))
content.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16719 entries, 0 to 16718
Data columns (total 29 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   OBJECTID    16719 non-null  int64  
 1   Id          0 non-null      float64
 2   Date        0 non-null      float64
 3   Time        0 non-null      float64
 4   DateTime    16719 non-null  object 
 5   ErrTime     16719 non-null  float64
 6   RMS         16719 non-null  float64
 7   Latitude    16719 non-null  float64
 8   Longitude   16719 non-null  float64
 9   Smaj        16719 non-null  float64
 10  Smin        16719 non-null  float64
 11  Az          16719 non-null  int64  
 12  Depth       16719 non-null  float64
 13  ErrDepth    16719 non-null  float64
 14  Ndef        16719 non-null  int64  
 15  Nsta        16719 non-null  int64  
 16  Gap         16719 non-null  int64  
 17  mdist       16719 non-null  float64
 18  Mdist_      16719 non-null  float64
 19  Qual        16719 non-nul

Una vez ejecutada la función `.info()` podemos empezar a operar sobre la estructura y las variables contenidas en el dataframe.

## 2. Descarte de variables

Lo primero que haremos será focalizar el contenido del dataframe en aquellas variables de interés para tener únicamente la información que vayamos a reflejar en el mapa de KeplerGL.

Para ello utilizamos la función drop, que nos permite seleccionar las columnas que vamos a descartar.

In [6]:
content.drop(['Id','OBJECTID','Date', 'Time','ErrTime','RMS','Smaj','Smin','Az','Depth','ErrDepth','Ndef','Nsta','Gap','mdist','Mdist_','Qual','Author','OrigID','IntensMax','Localizaci','TipoMagnit','XUTM','YUTM','GlobalID'], axis=1, inplace=True)

De esta forma tenemos únicamente cuatro métricas: magnitud, fecha, latitud y longitud

In [7]:
content

,DateTime,Latitude,Longitude,Magnitud
0,2017/10/07 10:27:29.820+00,28.5661,-17.8807,1.6
1,2017/10/07 13:03:25.460+00,28.5584,-17.8508,2.7
2,2017/10/07 20:09:58.790+00,28.5719,-17.8735,1.7
3,2017/10/07 23:06:00+00,28.5656,-17.8802,1.7
4,2017/10/08 01:23:02.150+00,28.5499,-17.8549,2.6
...,...,...,...,...
16714,2022/11/28 07:43:40.320+00,28.6084,-17.8826,1.3
16715,2022/11/29 21:02:30.530+00,28.6248,-17.8631,1.5
16716,2022/12/02 02:20:15.160+00,28.6213,-17.8763,1.4
16717,2022/12/02 04:08:59.920+00,28.6282,-17.8701,0.8


## 3. Selección de fechas

La base de datos comprende cinco años de registros, y para este ejercicio nos centraremos únicamente en los dos días previos a la erupción del volcán, el 19 de Septiembre de 2021.

Para ello haremos un doble condicional sobre el dataframe para definir la horquilla temporal para la cual queremos tener datos.

In [8]:
content=content[(content['DateTime'] > '2021/09/18 00:00:00.000+00') & (content['DateTime'] < '2021/09/20 00:00:00.000+00')]

De esta forma tenemos el dataframe delimitado para estas fechas, exclusivamente con 236 eventos sísmicos:

In [9]:
content

,DateTime,Latitude,Longitude,Magnitud
8441,2021/09/18 00:24:46.530+00,28.5699,-17.8678,1.4
8442,2021/09/18 01:06:06.710+00,28.5839,-17.8821,1.5
8443,2021/09/18 02:20:23.830+00,28.5775,-17.8927,2.1
8444,2021/09/18 02:30:45.890+00,28.5831,-17.8992,1.3
8445,2021/09/18 02:40:23.750+00,28.5726,-17.8712,1.9
...,...,...,...,...
8672,2021/09/19 18:29:48.730+00,28.6004,-17.9120,2.4
8673,2021/09/19 18:42:00.230+00,28.5661,-17.8630,2.5
8674,2021/09/19 18:43:21.080+00,28.6109,-17.9225,2.5
8675,2021/09/19 19:24:38.380+00,28.6138,-17.8956,2.6


## 4. Exportar en formato CSV

Una vez hemos seleccionado los eventos de interés, exportamos el dataframe en formato CSV para posteriormente incorporarlo a KeplerGL

In [10]:
content.to_csv(r'.\earthquakes_short_filtered.csv',index=False)